# Gate-logit trace extraction (Colab, GPU)

Reproducibility notebook for the WFPD paper. Loads an MoE model in 4-bit (NF4), runs WikiText-2, captures per-layer router logits, and saves them as `traces/{model}_gate_logits.npz` + a manifest. This is the GPU pipeline used to produce the paper's OLMoE-1B-7B traces; it complements `src/moe_routing/trace_extract.py` (the lightweight tiny-model default).

Run cells in order. The final cell zips and downloads the traces.

In [ ]:
!nvidia-smi -L
!pip install -q -U "transformers>=4.45" accelerate bitsandbytes datasets


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # avoid VRAM fragmentation
import json, re, gc, numpy as np, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

OUT = "traces"; os.makedirs(OUT, exist_ok=True)
NF4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

def get_texts(n_docs=128):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1",
                      split="train", streaming=True)
    out = []
    for ex in ds:
        t = ex["text"].strip()
        if len(t) >= 32: out.append(t)
        if len(out) >= n_docs: break
    return out

@torch.no_grad()
def extract(model_id, n_docs=128, max_tokens=128):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=NF4, device_map={"": 0},
        trust_remote_code=True).eval()
    model.config.output_router_logits = True
    buf = {}
    for text in get_texts(n_docs):
        ids = tok(text, return_tensors="pt", truncation=True,
                  max_length=max_tokens).to("cuda")
        rl = getattr(model(**ids, output_router_logits=True), "router_logits", None)
        if not rl:
            raise RuntimeError(f"{model_id}: no router_logits")
        for i, lg in enumerate(rl):
            if lg is None: continue
            buf.setdefault(i, []).append(
                lg.detach().float().reshape(-1, lg.shape[-1]).cpu().numpy().astype(np.float16))
    traces = {f"layer_{i:02d}": np.concatenate(v, 0) for i, v in sorted(buf.items()) if v}
    cfg = model.config
    ne = (getattr(cfg, "num_local_experts", None) or getattr(cfg, "num_experts", None)
          or next(iter(traces.values())).shape[1])
    man = {"model_id": model_id, "num_experts": int(ne),
           "top_k": int(getattr(cfg, "num_experts_per_tok", 0) or 0),
           "num_router_layers": len(traces),
           "tokens_per_layer": int(sum(a.shape[0] for a in traces.values()) // max(len(traces), 1)),
           "source": "Salesforce/wikitext (4-bit)", "layer_keys": sorted(traces)}
    tag = re.sub(r"[^0-9a-zA-Z]+", "_", model_id)
    np.savez_compressed(f"{OUT}/{tag}_gate_logits.npz", **traces)
    json.dump(man, open(f"{OUT}/{tag}_manifest.json", "w"), indent=2)
    print("OK", tag, "| experts", man["num_experts"], "top-k", man["top_k"],
          "| tokens/layer", man["tokens_per_layer"])
    del model, tok; gc.collect(); torch.cuda.empty_cache()   # free VRAM before next model

# The paper uses OLMoE-1B-7B. Uncomment the others to extract additional MoE families.
MODELS = [
    "allenai/OLMoE-1B-7B-0924",
    # "ibm-granite/granite-3.0-3b-a800m-base",
    # "ibm-granite/granite-3.0-1b-a400m-base",
]
for mid in MODELS:
    extract(mid)


In [ ]:
import shutil
shutil.make_archive("traces", "zip", "traces")
from google.colab import files
files.download("traces.zip")
